# Fig. 3(b) MSD Reproduction

This notebook reproduces the **shape and decoding workflow** of Fig. 3(b) from `arXiv:2412.15165v1`, using noisy `GeminiLogicalSimulator` runs of the d=3, 5-to-1 MSD circuit already encoded in `demo/msd.py`.

Paper-aligned choices used here:
- `reg[0]` is treated as the distilled output logical qubit.
- `reg[1:5]` are the four distillation-syndrome logical qubits.
- Factory acceptance is performed on the **corrected ancilla logical outcome `0011`**.
- The MLD is implemented as a lookup-table decoder learned from simulator data using the paper's special-state trick.
- The MLE decoder is implemented as a mixed-integer program over the detector error model with a logical-gap confidence score.

Notes:
- The paper uses very large training sample counts (up to `1e9`) for the MLD lookup table. This notebook exposes both fast and paper-scale settings, but defaults to a much smaller interactive configuration.
- The implementation uses the simulator's detector-error-model priors in place of external experimental calibrations.


In [2]:
import math
from collections import Counter, defaultdict
from dataclasses import dataclass
from functools import lru_cache
from itertools import product
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
from beliefmatching import detector_error_model_to_check_matrices
from bloqade import qubit, squin
from bloqade.gemini import logical as gemini_logical
from bloqade.lanes import GeminiLogicalSimulator
from bloqade.lanes.steane_defaults import steane7_m2dets, steane7_m2obs
from kirin.dialects import ilist
from scipy.optimize import Bounds, LinearConstraint, milp
from scipy.special import logsumexp


In [ ]:

FAST_CONFIG = {
    "mld_train_shots": 20_000,
    "eval_shots": 8_000,
    "posterior_samples": 20_000,
    "mle_threshold_points": 32,
}

PAPER_SCALE_CONFIG = {
    "mld_train_shots": 1_000_000_000,
    "eval_shots": 250_000,
    "posterior_samples": 200_000,
    "mle_threshold_points": 64,
}

CONFIG = FAST_CONFIG.copy()

BASIS_LABELS = ("X", "Y", "Z")
OUTPUT_QUBIT = 0
ANCILLA_QUBITS = (1, 2, 3, 4)
FACTORY_TARGET = np.array([0, 0, 1, 1], dtype=np.uint8)
SQRT3 = np.sqrt(3.0)

M2DETS_5 = steane7_m2dets(5)
M2OBS_5 = steane7_m2obs(5)
M2DETS_1 = steane7_m2dets(1)
M2OBS_1 = steane7_m2obs(1)

def bits_to_key(bits: np.ndarray | list[bool] | list[int]) -> str:
    return "".join("1" if int(x) else "0" for x in bits)

def key_to_bits(key: str) -> np.ndarray:
    return np.fromiter((1 if c == "1" else 0 for c in key), dtype=np.uint8)

def logical_expectation(bits: np.ndarray) -> float:
    if len(bits) == 0:
        return float("nan")
    return float(np.mean(1.0 - 2.0 * bits.astype(np.float64)))

def msd_magic_prep(reg):
    squin.broadcast.u3(0.3041 * math.pi, 0.25 * math.pi, 0.0, reg)

def msd_forward(reg):
    squin.broadcast.sqrt_x(ilist.IList([reg[0], reg[1], reg[4]]))
    squin.broadcast.cz(ilist.IList([reg[0], reg[2]]), ilist.IList([reg[1], reg[3]]))
    squin.broadcast.sqrt_y(ilist.IList([reg[0], reg[3]]))
    squin.broadcast.cz(ilist.IList([reg[0], reg[3]]), ilist.IList([reg[2], reg[4]]))
    squin.sqrt_x_adj(reg[0])
    squin.broadcast.cz(ilist.IList([reg[0], reg[1]]), ilist.IList([reg[4], reg[3]]))
    squin.broadcast.sqrt_y_adj(reg)

def msd_inverse(reg):
    squin.broadcast.sqrt_y(reg)
    squin.broadcast.cz(ilist.IList([reg[0], reg[1]]), ilist.IList([reg[4], reg[3]]))
    squin.sqrt_x(reg[0])
    squin.broadcast.cz(ilist.IList([reg[0], reg[3]]), ilist.IList([reg[2], reg[4]]))
    squin.broadcast.sqrt_y_adj(ilist.IList([reg[0], reg[3]]))
    squin.broadcast.cz(ilist.IList([reg[0], reg[2]]), ilist.IList([reg[1], reg[3]]))
    squin.broadcast.sqrt_x_adj(ilist.IList([reg[0], reg[1], reg[4]]))

def prepare_special_state(reg, basis: str):
    if basis == "X":
        squin.h(reg[OUTPUT_QUBIT])
    elif basis == "Y":
        squin.sqrt_x(reg[OUTPUT_QUBIT])
    elif basis == "Z":
        pass
    else:
        raise ValueError(basis)
    msd_inverse(reg)

def apply_tomography_rotations(reg, basis: str):
    squin.broadcast.h(ilist.IList([reg[i] for i in ANCILLA_QUBITS]))
    if basis == "X":
        return
    if basis == "Y":
        squin.sqrt_z_adj(reg[OUTPUT_QUBIT])
        return
    if basis == "Z":
        squin.h(reg[OUTPUT_QUBIT])
        return
    raise ValueError(basis)

def apply_injected_tomography_rotation(q, basis: str):
    if basis == "X":
        return
    if basis == "Y":
        squin.sqrt_z_adj(q)
        return
    if basis == "Z":
        squin.h(q)
        return
    raise ValueError(basis)

@gemini_logical.kernel(aggressive_unroll=True, verify=True)
def msd_actual_x():
    reg = qubit.qalloc(5)
    msd_magic_prep(reg)
    msd_forward(reg)
    apply_tomography_rotations(reg, "X")

@gemini_logical.kernel(aggressive_unroll=True, verify=True)
def msd_actual_y():
    reg = qubit.qalloc(5)
    msd_magic_prep(reg)
    msd_forward(reg)
    apply_tomography_rotations(reg, "Y")

@gemini_logical.kernel(aggressive_unroll=True, verify=True)
def msd_actual_z():
    reg = qubit.qalloc(5)
    msd_magic_prep(reg)
    msd_forward(reg)
    apply_tomography_rotations(reg, "Z")

@gemini_logical.kernel(aggressive_unroll=True, verify=True)
def msd_special_x():
    reg = qubit.qalloc(5)
    prepare_special_state(reg, "X")
    msd_forward(reg)
    apply_tomography_rotations(reg, "X")

@gemini_logical.kernel(aggressive_unroll=True, verify=True)
def msd_special_y():
    reg = qubit.qalloc(5)
    prepare_special_state(reg, "Y")
    msd_forward(reg)
    apply_tomography_rotations(reg, "Y")

@gemini_logical.kernel(aggressive_unroll=True, verify=True)
def msd_special_z():
    reg = qubit.qalloc(5)
    prepare_special_state(reg, "Z")
    msd_forward(reg)
    apply_tomography_rotations(reg, "Z")

@gemini_logical.kernel(aggressive_unroll=True, verify=True)
def injected_x():
    reg = qubit.qalloc(1)
    squin.u3(0.3041 * math.pi, 0.25 * math.pi, 0.0, reg[0])
    apply_injected_tomography_rotation(reg[0], "X")

@gemini_logical.kernel(aggressive_unroll=True, verify=True)
def injected_y():
    reg = qubit.qalloc(1)
    squin.u3(0.3041 * math.pi, 0.25 * math.pi, 0.0, reg[0])
    apply_injected_tomography_rotation(reg[0], "Y")

@gemini_logical.kernel(aggressive_unroll=True, verify=True)
def injected_z():
    reg = qubit.qalloc(1)
    squin.u3(0.3041 * math.pi, 0.25 * math.pi, 0.0, reg[0])
    apply_injected_tomography_rotation(reg[0], "Z")

ACTUAL_KERNELS = {"X": msd_actual_x, "Y": msd_actual_y, "Z": msd_actual_z}
SPECIAL_KERNELS = {"X": msd_special_x, "Y": msd_special_y, "Z": msd_special_z}
INJECTED_KERNELS = {"X": injected_x, "Y": injected_y, "Z": injected_z}


In [ ]:
@dataclass
class BasisDataset:
    detectors: np.ndarray
    observables: np.ndarray

def run_task(task, shots: int, with_noise: bool = True):
    result = task.run(shots, with_noise=with_noise, run_detectors=True)
    det = np.asarray(result.detectors, dtype=np.uint8)
    obs = np.asarray(result.observables, dtype=np.uint8)
    return BasisDataset(det, obs)

def split_factory_bits(detectors: np.ndarray, observables: np.ndarray):
    anc_det = detectors[:, 3:]
    anc_obs = observables[:, 1:]
    return anc_det, anc_obs

def build_mld_tables(dataset: BasisDataset):
    full_counts: dict[str, Counter[str]] = defaultdict(Counter)
    factory_counts: dict[str, Counter[str]] = defaultdict(Counter)
    output_flip_by_pattern: dict[str, list[int]] = defaultdict(list)

    anc_det, anc_obs = split_factory_bits(dataset.detectors, dataset.observables)
    for det, obs, a_det, a_obs in zip(dataset.detectors, dataset.observables, anc_det, anc_obs, strict=True):
        full_counts[bits_to_key(det)][bits_to_key(obs)] += 1
        factory_key = bits_to_key(a_det)
        factory_counts[factory_key][bits_to_key(a_obs)] += 1
        output_flip_by_pattern[factory_key].append(int(obs[0]))

    pattern_sign = {
        key: 1.0 - 2.0 * (sum(values) / len(values))
        for key, values in output_flip_by_pattern.items()
    }
    return {
        "full_counts": full_counts,
        "factory_counts": factory_counts,
        "pattern_sign": pattern_sign,
    }

def counter_argmax(counter: Counter[str], width: int) -> np.ndarray:
    if not counter:
        return np.zeros(width, dtype=np.uint8)
    best_key, _ = max(counter.items(), key=lambda kv: (kv[1], kv[0]))
    return key_to_bits(best_key)

def combine_pattern_scores(pattern_tables: dict[str, dict[str, Any]]) -> dict[str, float]:
    all_keys = set()
    for basis in BASIS_LABELS:
        all_keys.update(pattern_tables[basis]["pattern_sign"].keys())
    scores = {}
    for key in all_keys:
        basis_signs = []
        for basis in BASIS_LABELS:
            basis_signs.append(pattern_tables[basis]["pattern_sign"].get(key, 0.0))
        scores[key] = 0.5 + sum(basis_signs) / (2.0 * SQRT3)
    return scores

def rank_patterns(pattern_scores: dict[str, float]) -> list[str]:
    return [key for key, _ in sorted(pattern_scores.items(), key=lambda kv: (-kv[1], kv[0]))]

def compute_dem_data(task):
    dem_matrix = detector_error_model_to_check_matrices(
        task.detector_error_model,
        allow_undecomposed_hyperedges=True,
    )
    return {
        "H": dem_matrix.check_matrix.toarray().astype(np.int64),
        "O": dem_matrix.observables_matrix.toarray().astype(np.int64),
        "priors": np.asarray(dem_matrix.priors, dtype=np.float64),
    }

def _milp_score(H: np.ndarray, O: np.ndarray, priors: np.ndarray, syndrome: np.ndarray, logical_bits: np.ndarray):
    n_err = H.shape[1]
    n_det = H.shape[0]
    n_obs = O.shape[0]

    safe_priors = np.clip(priors, 1e-12, 1.0 - 1e-12)
    weights = np.log((1.0 - safe_priors) / safe_priors)

    A_det = np.hstack([H, -2 * np.eye(n_det, dtype=np.int64), np.zeros((n_det, n_obs), dtype=np.int64)])
    A_obs = np.hstack([O, np.zeros((n_obs, n_det), dtype=np.int64), -2 * np.eye(n_obs, dtype=np.int64)])
    A = np.vstack([A_det, A_obs])
    rhs = np.concatenate([syndrome.astype(np.int64), logical_bits.astype(np.int64)])
    bounds = Bounds(
        lb=np.concatenate([np.zeros(n_err, dtype=np.float64), np.zeros(n_det + n_obs, dtype=np.float64)]),
        ub=np.concatenate([np.ones(n_err, dtype=np.float64), np.full(n_det + n_obs, float(n_err), dtype=np.float64)]),
    )
    objective = np.concatenate([-weights, np.zeros(n_det + n_obs, dtype=np.float64)])
    integrality = np.ones(n_err + n_det + n_obs, dtype=np.uint8)
    result = milp(
        c=objective,
        integrality=integrality,
        bounds=bounds,
        constraints=LinearConstraint(A, rhs, rhs),
    )
    if not result.success:
        return None, -np.inf
    score = float(np.dot(weights, np.rint(result.x[:n_err]).astype(np.int64)))
    return np.rint(result.x[:n_err]).astype(np.uint8), score

def make_mle_decoder(task):
    dem = compute_dem_data(task)
    H = dem["H"]
    O = dem["O"]
    priors = dem["priors"]

    H_factory = H[3:, :]
    O_factory = O[1:, :]

    @lru_cache(maxsize=None)
    def decode_factory(syndrome_key: str):
        syndrome = key_to_bits(syndrome_key)
        scored = []
        for bits in product([0, 1], repeat=4):
            logical_bits = np.array(bits, dtype=np.uint8)
            _, score = _milp_score(H_factory, O_factory, priors, syndrome, logical_bits)
            if np.isfinite(score):
                scored.append((logical_bits, score))
        if not scored:
            return np.zeros(4, dtype=np.uint8), -np.inf
        scored.sort(key=lambda item: item[1], reverse=True)
        best_bits, best_score = scored[0]
        alt_score = next((score for bits, score in scored[1:] if not np.array_equal(bits, best_bits)), -np.inf)
        gap = float(best_score - alt_score) if np.isfinite(alt_score) else float("inf")
        return best_bits, gap

    @lru_cache(maxsize=None)
    def decode_full(syndrome_key: str):
        syndrome = key_to_bits(syndrome_key)
        scored = []
        for bits in product([0, 1], repeat=5):
            logical_bits = np.array(bits, dtype=np.uint8)
            _, score = _milp_score(H, O, priors, syndrome, logical_bits)
            if np.isfinite(score):
                scored.append((logical_bits, score))
        if not scored:
            return np.zeros(5, dtype=np.uint8)
        return max(scored, key=lambda item: item[1])[0]

    return decode_factory, decode_full

def weighted_quantile(values: np.ndarray, quantiles: list[float], weights: np.ndarray):
    order = np.argsort(values)
    values = values[order]
    weights = weights[order]
    cdf = np.cumsum(weights)
    cdf /= cdf[-1]
    return np.interp(quantiles, cdf, values)

def sample_bloch_ball(num_samples: int, seed: int = 1234) -> np.ndarray:
    rng = np.random.default_rng(seed)
    directions = rng.normal(size=(num_samples, 3))
    directions /= np.linalg.norm(directions, axis=1, keepdims=True)
    radii = rng.random(num_samples) ** (1.0 / 3.0)
    return directions * radii[:, None]

def fidelity_from_counts(x_bits: np.ndarray, y_bits: np.ndarray, z_bits: np.ndarray, posterior_samples: int):
    ex = logical_expectation(x_bits)
    ey = logical_expectation(y_bits)
    ez = logical_expectation(z_bits)
    point = 0.5 + (ex + ey + ez) / (2.0 * SQRT3)

    n = np.array([len(x_bits), len(y_bits), len(z_bits)], dtype=np.int64)
    k = np.array([
        int(np.sum(x_bits == 0)),
        int(np.sum(y_bits == 0)),
        int(np.sum(z_bits == 0)),
    ], dtype=np.int64)

    points = sample_bloch_ball(posterior_samples)
    probs = np.clip((1.0 + points) / 2.0, 1e-12, 1.0 - 1e-12)
    log_weights = (k * np.log(probs) + (n - k) * np.log1p(-probs)).sum(axis=1)
    weights = np.exp(log_weights - logsumexp(log_weights))
    fidelities = 0.5 + np.sum(points, axis=1) / (2.0 * SQRT3)
    q16, q50, q84 = weighted_quantile(fidelities, [0.16, 0.5, 0.84], weights)
    return {
        "point": float(point),
        "median": float(q50),
        "low": float(q16),
        "high": float(q84),
        "bloch": (float(ex), float(ey), float(ez)),
    }

def evaluate_mld_curve(actual_data, lookup_tables, ranked_patterns, posterior_samples: int):
    accepted_fractions = []
    fidelities = []
    credibility = []

    for keep_count in range(1, len(ranked_patterns) + 1):
        kept = set(ranked_patterns[:keep_count])
        corrected = {}
        total_kept = 0
        total_shots = 0
        for basis in BASIS_LABELS:
            dataset = actual_data[basis]
            anc_det, anc_obs = split_factory_bits(dataset.detectors, dataset.observables)
            corrected_bits = []
            for det, obs, a_det, a_obs in zip(dataset.detectors, dataset.observables, anc_det, anc_obs, strict=True):
                factory_table = lookup_tables[basis]["factory_counts"]
                full_table = lookup_tables[basis]["full_counts"]
                anc_key = bits_to_key(a_det)
                anc_flip = counter_argmax(factory_table.get(anc_key, Counter()), 4)
                corrected_anc = a_obs ^ anc_flip
                if not np.array_equal(corrected_anc, FACTORY_TARGET):
                    continue
                if anc_key not in kept:
                    continue
                full_key = bits_to_key(det)
                full_flip = counter_argmax(full_table.get(full_key, Counter()), 5)
                corrected_bits.append(int(obs[0] ^ full_flip[0]))
            corrected[basis] = np.asarray(corrected_bits, dtype=np.uint8)
            total_kept += len(corrected[basis])
            total_shots += len(dataset.observables)
        if min(len(corrected[b]) for b in BASIS_LABELS) == 0:
            continue
        accepted_fractions.append(total_kept / total_shots)
        summary = fidelity_from_counts(corrected["X"], corrected["Y"], corrected["Z"], posterior_samples)
        fidelities.append(summary["point"])
        credibility.append((summary["low"], summary["high"]))
    return {
        "accepted_fraction": np.asarray(accepted_fractions, dtype=np.float64),
        "fidelity": np.asarray(fidelities, dtype=np.float64),
        "credible": np.asarray(credibility, dtype=np.float64),
    }

def evaluate_mle_curve(actual_data, decode_factory, decode_full, posterior_samples: int, threshold_points: int):
    all_gaps = []
    for basis in BASIS_LABELS:
        dataset = actual_data[basis]
        anc_det, anc_obs = split_factory_bits(dataset.detectors, dataset.observables)
        for a_det, a_obs in zip(anc_det, anc_obs, strict=True):
            anc_flip, gap = decode_factory(bits_to_key(a_det))
            if np.array_equal(a_obs ^ anc_flip, FACTORY_TARGET) and np.isfinite(gap):
                all_gaps.append(gap)
    if not all_gaps:
        raise RuntimeError("No factory-accepted shots found for MLE threshold sweep")
    thresholds = np.quantile(np.asarray(all_gaps), np.linspace(0.0, 1.0, threshold_points))

    accepted_fractions = []
    fidelities = []
    credibility = []
    for threshold in thresholds:
        corrected = {}
        total_kept = 0
        total_shots = 0
        for basis in BASIS_LABELS:
            dataset = actual_data[basis]
            anc_det, anc_obs = split_factory_bits(dataset.detectors, dataset.observables)
            corrected_bits = []
            for det, obs, a_det, a_obs in zip(dataset.detectors, dataset.observables, anc_det, anc_obs, strict=True):
                anc_flip, gap = decode_factory(bits_to_key(a_det))
                corrected_anc = a_obs ^ anc_flip
                if not np.array_equal(corrected_anc, FACTORY_TARGET):
                    continue
                if gap < threshold:
                    continue
                full_flip = decode_full(bits_to_key(det))
                corrected_bits.append(int(obs[0] ^ full_flip[0]))
            corrected[basis] = np.asarray(corrected_bits, dtype=np.uint8)
            total_kept += len(corrected[basis])
            total_shots += len(dataset.observables)
        if min(len(corrected[b]) for b in BASIS_LABELS) == 0:
            continue
        accepted_fractions.append(total_kept / total_shots)
        summary = fidelity_from_counts(corrected["X"], corrected["Y"], corrected["Z"], posterior_samples)
        fidelities.append(summary["point"])
        credibility.append((summary["low"], summary["high"]))
    return {
        "accepted_fraction": np.asarray(accepted_fractions, dtype=np.float64),
        "fidelity": np.asarray(fidelities, dtype=np.float64),
        "credible": np.asarray(credibility, dtype=np.float64),
    }

def injected_baseline(task_map, posterior_samples: int):
    corrected = {}
    for basis in BASIS_LABELS:
        task = task_map[basis]
        dataset = run_task(task, CONFIG["eval_shots"], with_noise=True)
        table = build_mld_tables(dataset)
        bits = []
        for det, obs in zip(dataset.detectors, dataset.observables, strict=True):
            flip = counter_argmax(table["full_counts"].get(bits_to_key(det), Counter()), 1)
            bits.append(int(obs[0] ^ flip[0]))
        corrected[basis] = np.asarray(bits, dtype=np.uint8)
    return fidelity_from_counts(corrected["X"], corrected["Y"], corrected["Z"], posterior_samples)


In [ ]:
sim = GeminiLogicalSimulator()

actual_tasks = {basis: sim.task(kernel, m2dets=M2DETS_5, m2obs=M2OBS_5) for basis, kernel in ACTUAL_KERNELS.items()}
special_tasks = {basis: sim.task(kernel, m2dets=M2DETS_5, m2obs=M2OBS_5) for basis, kernel in SPECIAL_KERNELS.items()}
injected_tasks = {basis: sim.task(kernel, m2dets=M2DETS_1, m2obs=M2OBS_1) for basis, kernel in INJECTED_KERNELS.items()}

print("Compiled tasks:")
print(" actual:", list(actual_tasks))
print(" special:", list(special_tasks))
print(" injected:", list(injected_tasks))

In [ ]:
# Validation: special-state kernels should decode to all-zero logical error strings in the noiseless limit.
for basis, task in special_tasks.items():
    data = run_task(task, 256, with_noise=False)
    print(basis, "special-state noiseless observables:", sorted({tuple(row) for row in data.observables}))

# Validation: the factory acceptance should be close to 1/6 in the noiseless limit after decoding ancilla outcomes.
noiseless_actual = {basis: run_task(task, 2048, with_noise=False) for basis, task in actual_tasks.items()}
for basis, dataset in noiseless_actual.items():
    anc_obs = dataset.observables[:, 1:]
    acceptance = np.mean(np.all(anc_obs == FACTORY_TARGET, axis=1))
    print(basis, "raw noiseless factory acceptance (uncorrected ancilla observables):", acceptance)


In [ ]:
mld_training = {}
for basis, task in special_tasks.items():
    print(f"Sampling MLD training data for {basis} with {CONFIG['mld_train_shots']:,} shots...")
    dataset = run_task(task, CONFIG["mld_train_shots"], with_noise=True)
    mld_training[basis] = build_mld_tables(dataset)

pattern_scores = combine_pattern_scores(mld_training)
ranked_patterns = rank_patterns(pattern_scores)
print("Unique factory patterns ranked:", len(ranked_patterns))

actual_data = {
    basis: run_task(task, CONFIG["eval_shots"], with_noise=True)
    for basis, task in actual_tasks.items()
}

decode_factory, decode_full = make_mle_decoder(actual_tasks["X"])

mld_curve = evaluate_mld_curve(actual_data, mld_training, ranked_patterns, CONFIG["posterior_samples"])
mle_curve = evaluate_mle_curve(
    actual_data,
    decode_factory,
    decode_full,
    CONFIG["posterior_samples"],
    CONFIG["mle_threshold_points"],
)
injected_summary = injected_baseline(injected_tasks, CONFIG["posterior_samples"])

print("Injected baseline fidelity:", injected_summary["point"])
print("MLE curve points:", len(mle_curve["accepted_fraction"]))
print("MLD curve points:", len(mld_curve["accepted_fraction"]))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

ax.plot(mle_curve["accepted_fraction"], mle_curve["fidelity"], color="tab:blue", label="Distilled (MLE)")
ax.fill_between(
    mle_curve["accepted_fraction"],
    mle_curve["credible"][:, 0],
    mle_curve["credible"][:, 1],
    color="tab:blue",
    alpha=0.15,
)

ax.plot(mld_curve["accepted_fraction"], mld_curve["fidelity"], color="tab:orange", label="Distilled (MLD)")
ax.fill_between(
    mld_curve["accepted_fraction"],
    mld_curve["credible"][:, 0],
    mld_curve["credible"][:, 1],
    color="tab:orange",
    alpha=0.15,
)

ax.axhline(injected_summary["point"], color="tab:green", linestyle="-", label="Injected")
ax.fill_between(
    [0.0, 1.0],
    injected_summary["low"],
    injected_summary["high"],
    color="tab:green",
    alpha=0.1,
)

ax.set_xlabel("Total accepted fraction")
ax.set_ylabel("Magic state fidelity")
ax.set_title("Reproduction of Fig. 3(b) on GeminiLogicalSimulator")
ax.grid(alpha=0.25)
ax.legend()
plt.show()